## Create schema-s for 3 layers

In [0]:
%python
# Create schemas for a task
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## Load raw data into bronze layer as tables: 
- bronze.items_raw
- bronze.events_raw

### Observations
- The dataset has been transferred with string types as requested.

### Item Data
- There are null values in the `adjective` and `modifier` columns; a rule for handling or populating these could be introduced based on agreement.
- The `id` column contains float values, although it should essentially be of type int.
- The `price` column is of float type, but since it represents monetary values, it could potentially be rounded to 2 decimal places.

### Event Data
- `event.payload` has not been fully transferred; the load process needs to be adjusted so that this field, which is inherently of JSON type, can be properly ingested.
- `user_id` is of float type, while it should naturally be int.
- Other columns are strings.



In [0]:
%python
# Load dataframes into a bronze tables
item_path = "s3://merkle-de-interview-case-study/de/item.csv"
event_path = "s3://merkle-de-interview-case-study/de/event.csv"

item_df = spark.read.option("header", True).csv(item_path)
event_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .load(event_path)
)
# write data in tables
item_df.write.mode("overwrite").saveAsTable("bronze.items_raw")
event_df.write.mode("overwrite").saveAsTable("bronze.events_raw")
item_df, event_df = None, None

## Clean up and transform data for silver tables
#### Clean up item data `bronze.items_raw` to `silver.items`
due to observations in the data:
- I would suggest defining a standard or agreement for handling `null` categorical values; in this case, I will use `Unknown`.
- Additionally, an agreement should be established regarding the rounding of `price` values.





In [0]:
from pyspark.sql.functions import coalesce, lit, col, round, to_timestamp

items_transformed_df = spark.table("bronze.items_raw") \
    .withColumn("adjective", coalesce(col("adjective"), lit("Unknown"))) \
    .withColumn("modifier", coalesce(col("modifier"), lit("Unknown"))) \
    .withColumn("created_at", to_timestamp(col("created_at"))) \
    .withColumn("id", col("id").cast("double").cast("int")) \
    .withColumn("price", round(col("price").cast("double"), 2)) 

# Save items_transformed_df to silver.items table
items_transformed_df.write.format("delta").mode("overwrite").saveAsTable("silver.items")
items_transformed_df = None


#### Clean up event data `bronze.events_raw` to `silver.events`
due to observations in the data:
Several steps are required:
- casting data types of the base columns and assigning aliases  
- flattening JSON records in the `payload`  
- pivoting `parameter_name` with their corresponding values into separate columns (this results in the uniqueness of `event_id`)  
- casting the newly created columns  

An open question remains regarding test rows — an agreement is needed on whether they should be cleaned/removed or retained.





In [0]:
from pyspark.sql.functions import col, from_json, first, year
from pyspark.sql.types import StructType, StringType

# Cast and rename columns for bronze.events_raw table, as a first stap
bronze_event_df = spark.table("bronze.events_raw") \
    .withColumn("event_id", col("event_id").cast("string")) \
    .withColumn("event_time", col("event_time").cast("timestamp")) \
    .withColumn("user_id", col("user_id").cast("double").cast("int")) \
    .withColumnRenamed("event.payload", "event_payload")


# Define schema for event_payload JSON as a step for parse JSON
event_payload_schema = StructType() \
    .add("event_name", StringType()) \
    .add("platform", StringType()) \
    .add("parameter_name", StringType()) \
    .add("parameter_value", StringType())

# Parse event_payload column
bronze_event_flat_df = bronze_event_df.withColumn(
    "event_payload_json", from_json(col("event_payload"), event_payload_schema)
).select(
    "event_id",
    "event_time",
    "user_id",
    "event_payload_json.event_name",
    "event_payload_json.platform",
    "event_payload_json.parameter_name",
    "event_payload_json.parameter_value"
)

# Pivot parameter_name values to columns, making event_id unique
bronze_event_pivot_df = bronze_event_flat_df.groupBy(
    "event_id", "event_time", "user_id", "event_name", "platform"
).pivot("parameter_name").agg(first("parameter_value"))


# additional cast over pivoted columns
bronze_event_pivot_casted_df = bronze_event_pivot_df \
    .withColumn("item_id", col("item_id").cast("int")) \
    .withColumn("test_assignment", col("test_assignment").cast("int")) \
    .withColumn("test_id", col("test_id").cast("int")) \
    .withColumn("viewed_user_id", col("viewed_user_id").cast("int")) \
    .withColumn("year", year(col("event_time")))

# Save bronze_event_pivot_casted_df to silver.events table partitioned by year
bronze_event_pivot_casted_df.write.format("delta").mode("overwrite").partitionBy("year").saveAsTable("silver.events")

bronze_event_df = None
bronze_event_flat_df = None
bronze_event_pivot_df = None
bronze_event_pivot_casted_df = None


##Gold layer
### Summary of Observations and Approach

- The `events` dataset is a strong candidate for a fact table, while `items` is a candidate for a dimension table.

- To properly model `items` as a dimension, an approach similar to SCD Type 2 would be required. Although the current dataset has uniqueness on the natural key (`id`), this may not hold in the future. In such a case, a surrogate key would be needed, for example as a hash of the `id` combined with the validity period (`from`, `to`).

- To fully model `events` as a fact table, it should ideally be cleansed of categorical attributes (mostly originating from the processed JSON `payload`) and potentially moved into a junk dimension, also managed with SCD Type 2. The `fact_events` table would then rely on surrogate keys referencing dimensions, while possibly retaining a timestamp column. However, since categorical attributes do not significantly impact memory usage, they can remain in the fact table to avoid unnecessary complexity.

- The `events` table also contains a `user_id` field, which could be leveraged if a `dim_user` table exists, or for analyzing user behavior without explicitly modeling a user dimension.

- For this task, I will proceed as a POC and model `events` as a fact table and `items` as a dimension, linking them through their natural keys.

In [0]:
# create from silver.items to gold.dim_items
spark.table("silver.items") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_items")

# create from silver.events to gold.fact_events
spark.table("silver.events") \
    .write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year") \
    .saveAsTable("gold.fact_events")

#### creation of data mart gold.top_items

In [0]:
top_item_df = spark.sql("""
WITH prep AS 
(
    SELECT 
        i.name
        , e.year
        , e.platform
        , COUNT(*) AS cnt
    FROM gold.fact_events e
    JOIN gold.dim_items i
        ON e.item_id = i.id
    WHERE e.event_name = 'view_item'
    GROUP BY 
        i.name
        , e.year
        , e.platform
    ORDER BY 
        i.name
        , e.year
        , e.platform
)
, prep2 AS 
(
    SELECT 
        name AS item_name
        , year
        , platform
        , cnt
        , row_number() OVER (PARTITION BY name, year ORDER BY cnt DESC) AS rn_platform -- used to filter onli rank 1
        , sum(cnt) OVER (PARTITION BY name, year) AS total_views -- Total number of item views in a particular year
    FROM prep
)
SELECT 
    item_name
    , year
    , total_views -- Total number of item views in a particular year.
    , platform AS most_used_platform   -- Platform on which the item was viewed the most
    , dense_rank() OVER (
        PARTITION BY year 
        ORDER BY total_views DESC
        ) AS rank_of_item_per_year -- Rank of the item in the year
FROM prep2
WHERE rn_platform = 1
""")

top_item_df.write.format("delta").mode("overwrite").partitionBy("year").saveAsTable("gold.top_items")
top_item_df = None
